In [3]:
import os

# Paths
train_images_path = "kfu data/train images"
train_masks_path  = "kfu data/train masks"
test_images_path  = "kfu data/test images"

# Counts
print("Train images:", len(os.listdir(train_images_path)))
print("Train masks: ", len(os.listdir(train_masks_path)))
print("Test images: ", len(os.listdir(test_images_path)))

Train images: 2000
Train masks:  2001
Test images:  2000


In [4]:
# ======================================================
# LeViT + UNet++ (FINAL OPTIMIZED VERSION)
# ======================================================

import os, cv2, random, gc
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, average_precision_score
from tqdm import tqdm
import albumentations as A
import timm

# ------------------
# SEED & DEVICE
# ------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# ------------------
# CONFIG (FINAL SAFE)
# ------------------
IMG_SIZE = 224        # MUST for LeViT
BATCH_SIZE = 2        # safe for GPU
EPOCHS = 50
AUG_MULTIPLIER = 3    # faster training

train_images_path = "kfu data/train images"
train_masks_path  = "kfu data/train masks"

# ------------------
# AUGMENTATION (BETTER BUT LIGHT)
# ------------------
train_aug = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=20, p=0.5),
    A.RandomBrightnessContrast(p=0.2),
])

# ======================================================
# DATASET
# ======================================================
class DatasetSeg(Dataset):
    def __init__(self, ids, augment=False, multiplier=1):
        self.ids = ids * multiplier
        self.augment = augment

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        name = self.ids[idx]

        img = cv2.imread(os.path.join(train_images_path, name + ".jpg"))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

        mask = cv2.imread(os.path.join(train_masks_path, name + ".png"), 0)
        mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
        mask = (mask == 255).astype(np.float32)

        if self.augment:
            aug = train_aug(image=img, mask=mask)
            img, mask = aug["image"], aug["mask"]

        img = img / 255.0
        img = np.transpose(img, (2,0,1))

        return torch.tensor(img, dtype=torch.float32), torch.tensor(mask).unsqueeze(0)

# ------------------
# SPLIT
# ------------------
ids = [os.path.splitext(f)[0] for f in os.listdir(train_images_path)]
train_ids, val_ids = train_test_split(ids, test_size=0.2, random_state=SEED)

train_ds = DatasetSeg(train_ids, augment=True, multiplier=AUG_MULTIPLIER)
val_ds   = DatasetSeg(val_ids, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

print("Train samples:", len(train_ds))
print("Val samples  :", len(val_ds))

# ======================================================
# MODEL (OPTIMIZED DECODER)
# ======================================================
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU()
        )
    def forward(self, x): return self.conv(x)

class Up(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, 2, 2)
    def forward(self, x): return self.up(x)

class LeViT_UNetPP(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = timm.create_model(
            "levit_128",
            pretrained=True,
            features_only=True,
            img_size=224
        )

        chs = self.encoder.feature_info.channels()
        print("Encoder channels:", chs)

        self.conv1 = ConvBlock(chs[-1], 128)

        self.up1 = Up(128,64)
        self.conv2 = ConvBlock(64 + chs[-2],64)

        self.up2 = Up(64,32)
        self.conv3 = ConvBlock(32 + chs[-3],32)

        self.out = nn.Conv2d(32,1,1)

    def forward(self, x):
        size = x.shape[2:]

        x1, x2, x3 = self.encoder(x)

        d = self.conv1(x3)

        d = self.up1(d)
        x2 = F.interpolate(x2, size=d.shape[2:], mode="bilinear", align_corners=False)
        d = self.conv2(torch.cat([d, x2], 1))

        d = self.up2(d)
        x1 = F.interpolate(x1, size=d.shape[2:], mode="bilinear", align_corners=False)
        d = self.conv3(torch.cat([d, x1], 1))

        out = self.out(d)
        out = F.interpolate(out, size=size, mode="bilinear", align_corners=False)

        return out

model = LeViT_UNetPP().to(device)

# ======================================================
# LOSS
# ======================================================
bce = nn.BCEWithLogitsLoss()

def dice_loss(pred, target):
    pred = torch.sigmoid(pred)
    smooth = 1e-7
    return 1 - (2*(pred*target).sum() + smooth)/(pred.sum()+target.sum()+smooth)

def loss_fn(pred, target):
    return bce(pred, target) + dice_loss(pred, target)

optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)

scaler = torch.amp.GradScaler("cuda")

# ======================================================
# METRICS
# ======================================================
def compute_metrics(P, M):
    P = np.clip(P, -20, 20)
    P = 1 / (1 + np.exp(-P))

    P_bin = (P > 0.5).astype(np.uint8)
    M = (M > 0.5).astype(np.uint8)

    P_bin = P_bin.flatten()
    M = M.flatten()

    acc = accuracy_score(M, P_bin)
    prec = precision_score(M, P_bin, zero_division=0)
    rec = recall_score(M, P_bin, zero_division=0)
    f1 = f1_score(M, P_bin, zero_division=0)

    inter = (P_bin * M).sum()
    dice = (2*inter)/(P_bin.sum()+M.sum()+1e-7)
    iou = inter/(P_bin.sum()+M.sum()-inter+1e-7)

    ap50 = average_precision_score(M, P)

    return acc, prec, rec, f1, dice, iou, ap50

# ======================================================
# TRAIN
# ======================================================
print("\nTraining...\n")

best_dice = 0

for epoch in range(EPOCHS):
    model.train()
    preds_all, masks_all = [], []

    for imgs, masks in tqdm(train_loader, leave=False):
        imgs, masks = imgs.to(device), masks.to(device)

        with torch.amp.autocast("cuda"):
            preds = model(imgs)
            loss = loss_fn(preds, masks)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        preds_all.append(preds.detach().cpu().numpy())
        masks_all.append(masks.cpu().numpy())

    P = np.concatenate(preds_all).flatten()
    M = np.concatenate(masks_all).flatten()

    acc,prec,rec,f1,dice,iou,ap50 = compute_metrics(P,M)

    print(f"Epoch {epoch+1}/{EPOCHS} | Acc:{acc:.4f} Prec:{prec:.4f} Rec:{rec:.4f} F1:{f1:.4f} Dice:{dice:.4f} IoU:{iou:.4f} mAP50:{ap50:.4f}")

    # Save best model
    if dice > best_dice:
        best_dice = dice
        torch.save(model.state_dict(), "best_model.pth")

    torch.cuda.empty_cache()
    gc.collect()

# ======================================================
# FINAL TEST
# ======================================================
model.load_state_dict(torch.load("best_model.pth"))

model.eval()
preds_all, masks_all = [], []

with torch.no_grad():
    for imgs, masks in val_loader:
        imgs = imgs.to(device)
        masks = masks.to(device)

        preds = model(imgs)

        preds_all.append(preds.cpu().numpy())
        masks_all.append(masks.cpu().numpy())

P = np.concatenate(preds_all).flatten()
M = np.concatenate(masks_all).flatten()

acc,prec,rec,f1,dice,iou,ap50 = compute_metrics(P,M)

print("\n================ FINAL TEST RESULTS ================")
print(f"Accuracy     : {acc:.4f}")
print(f"Precision    : {prec:.4f}")
print(f"Recall       : {rec:.4f}")
print(f"F1-score     : {f1:.4f}")
print(f"Dice         : {dice:.4f}")
print(f"IoU          : {iou:.4f}")
print(f"mAP@50       : {ap50:.4f}")
print("===================================================")

Using device: cuda
Train samples: 4800
Val samples  : 400
Encoder channels: [128, 256, 384]

Training...



Epoch 1/50 | Acc:0.9809 Prec:0.5643 Rec:0.6149 F1:0.5885 Dice:0.5885 IoU:0.4169 mAP50:0.5566


Epoch 2/50 | Acc:0.9872 Prec:0.7265 Rec:0.6768 F1:0.7008 Dice:0.7008 IoU:0.5394 mAP50:0.7394


Epoch 3/50 | Acc:0.9885 Prec:0.7542 Rec:0.7157 F1:0.7344 Dice:0.7344 IoU:0.5803 mAP50:0.7781


Epoch 4/50 | Acc:0.9894 Prec:0.7713 Rec:0.7394 F1:0.7550 Dice:0.7550 IoU:0.6064 mAP50:0.8072


...

Epoch 46/50 | Acc:0.9951 Prec:0.8869 Rec:0.8926 F1:0.8897 Dice:0.8897 IoU:0.8014 mAP50:0.9515


Epoch 47/50 | Acc:0.9951 Prec:0.8891 Rec:0.8908 F1:0.8900 Dice:0.8900 IoU:0.8018 mAP50:0.9517


Epoch 48/50 | Acc:0.9951 Prec:0.8895 Rec:0.8920 F1:0.8907 Dice:0.8907 IoU:0.8030 mAP50:0.9530


Epoch 49/50 | Acc:0.9952 Prec:0.8898 Rec:0.8932 F1:0.8915 Dice:0.8915 IoU:0.8042 mAP50:0.9530


Epoch 50/50 | Acc:0.9951 Prec:0.8890 Rec:0.8921 F1:0.8906 Dice:0.8906 IoU:0.8027 mAP50:0.9527

================ FINAL TEST RESULTS ================
Accuracy     : 0.9908
Precision    : 0.8606
Recall       : 0.7741
F1-score     : 0.8151
Dice         : 0.8151
IoU          : 0.6878
mAP@50       : 0.8798


In [5]:
# ======================================================
# LeViT + UNet++ (FINAL OPTIMIZED VERSION)
# ======================================================

import os, cv2, random, gc
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, average_precision_score
from tqdm import tqdm
import albumentations as A
import timm

# ------------------
# SEED & DEVICE
# ------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# ------------------
# CONFIG (FINAL SAFE)
# ------------------
IMG_SIZE = 224        # MUST for LeViT
BATCH_SIZE = 2        # safe for GPU
EPOCHS = 50
AUG_MULTIPLIER = 3    # faster training

train_images_path = "kfu data/train images"
train_masks_path  = "kfu data/train masks"

# ------------------
# AUGMENTATION (BETTER BUT LIGHT)
# ------------------
train_aug = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=20, p=0.5),
    A.RandomBrightnessContrast(p=0.2),
])

# ======================================================
# DATASET
# ======================================================
class DatasetSeg(Dataset):
    def __init__(self, ids, augment=False, multiplier=1):
        self.ids = ids * multiplier
        self.augment = augment

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        name = self.ids[idx]

        img = cv2.imread(os.path.join(train_images_path, name + ".jpg"))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

        mask = cv2.imread(os.path.join(train_masks_path, name + ".png"), 0)
        mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
        mask = (mask == 255).astype(np.float32)

        if self.augment:
            aug = train_aug(image=img, mask=mask)
            img, mask = aug["image"], aug["mask"]

        img = img / 255.0
        img = np.transpose(img, (2,0,1))

        return torch.tensor(img, dtype=torch.float32), torch.tensor(mask).unsqueeze(0)

# ------------------
# SPLIT
# ------------------
ids = [os.path.splitext(f)[0] for f in os.listdir(train_images_path)]
train_ids, val_ids = train_test_split(ids, test_size=0.2, random_state=SEED)

train_ds = DatasetSeg(train_ids, augment=True, multiplier=AUG_MULTIPLIER)
val_ds   = DatasetSeg(val_ids, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

print("Train samples:", len(train_ds))
print("Val samples  :", len(val_ds))

# ======================================================
# MODEL (OPTIMIZED DECODER)
# ======================================================
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU()
        )
    def forward(self, x): return self.conv(x)

class Up(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, 2, 2)
    def forward(self, x): return self.up(x)

class LeViT_UNetPP(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = timm.create_model(
            "levit_128",
            pretrained=True,
            features_only=True,
            img_size=224
        )

        chs = self.encoder.feature_info.channels()
        print("Encoder channels:", chs)

        self.conv1 = ConvBlock(chs[-1], 128)

        self.up1 = Up(128,64)
        self.conv2 = ConvBlock(64 + chs[-2],64)

        self.up2 = Up(64,32)
        self.conv3 = ConvBlock(32 + chs[-3],32)

        self.out = nn.Conv2d(32,1,1)

    def forward(self, x):
        size = x.shape[2:]

        x1, x2, x3 = self.encoder(x)

        d = self.conv1(x3)

        d = self.up1(d)
        x2 = F.interpolate(x2, size=d.shape[2:], mode="bilinear", align_corners=False)
        d = self.conv2(torch.cat([d, x2], 1))

        d = self.up2(d)
        x1 = F.interpolate(x1, size=d.shape[2:], mode="bilinear", align_corners=False)
        d = self.conv3(torch.cat([d, x1], 1))

        out = self.out(d)
        out = F.interpolate(out, size=size, mode="bilinear", align_corners=False)

        return out

model = LeViT_UNetPP().to(device)

# ======================================================
# LOSS
# ======================================================
bce = nn.BCEWithLogitsLoss()

def dice_loss(pred, target):
    pred = torch.sigmoid(pred)
    smooth = 1e-7
    return 1 - (2*(pred*target).sum() + smooth)/(pred.sum()+target.sum()+smooth)

def loss_fn(pred, target):
    return bce(pred, target) + dice_loss(pred, target)

optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)

scaler = torch.amp.GradScaler("cuda")

# ======================================================
# METRICS
# ======================================================
def compute_metrics(P, M):
    P = np.clip(P, -20, 20)
    P = 1 / (1 + np.exp(-P))

    P_bin = (P > 0.5).astype(np.uint8)
    M = (M > 0.5).astype(np.uint8)

    P_bin = P_bin.flatten()
    M = M.flatten()

    acc = accuracy_score(M, P_bin)
    prec = precision_score(M, P_bin, zero_division=0)
    rec = recall_score(M, P_bin, zero_division=0)
    f1 = f1_score(M, P_bin, zero_division=0)

    inter = (P_bin * M).sum()
    dice = (2*inter)/(P_bin.sum()+M.sum()+1e-7)
    iou = inter/(P_bin.sum()+M.sum()-inter+1e-7)

    ap50 = average_precision_score(M, P)

    return acc, prec, rec, f1, dice, iou, ap50

# ======================================================
# TRAIN
# ======================================================
print("\nTraining...\n")

best_dice = 0

for epoch in range(EPOCHS):
    model.train()
    preds_all, masks_all = [], []

    for imgs, masks in tqdm(train_loader, leave=False):
        imgs, masks = imgs.to(device), masks.to(device)

        with torch.amp.autocast("cuda"):
            preds = model(imgs)
            loss = loss_fn(preds, masks)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        preds_all.append(preds.detach().cpu().numpy())
        masks_all.append(masks.cpu().numpy())

    P = np.concatenate(preds_all).flatten()
    M = np.concatenate(masks_all).flatten()

    acc,prec,rec,f1,dice,iou,ap50 = compute_metrics(P,M)

    print(f"Epoch {epoch+1}/{EPOCHS} | Acc:{acc:.4f} Prec:{prec:.4f} Rec:{rec:.4f} F1:{f1:.4f} Dice:{dice:.4f} IoU:{iou:.4f} mAP50:{ap50:.4f}")

    # Save best model
    if dice > best_dice:
        best_dice = dice
        torch.save(model.state_dict(), "best_model.pth")

    torch.cuda.empty_cache()
    gc.collect()

# ======================================================
# FINAL TEST
# ======================================================
model.load_state_dict(torch.load("best_model.pth"))

model.eval()
preds_all, masks_all = [], []

with torch.no_grad():
    for imgs, masks in val_loader:
        imgs = imgs.to(device)
        masks = masks.to(device)

        preds = model(imgs)

        preds_all.append(preds.cpu().numpy())
        masks_all.append(masks.cpu().numpy())

P = np.concatenate(preds_all).flatten()
M = np.concatenate(masks_all).flatten()

acc,prec,rec,f1,dice,iou,ap50 = compute_metrics(P,M)

print("\n================ FINAL TEST RESULTS ================")
print(f"Accuracy     : {acc:.4f}")
print(f"Precision    : {prec:.4f}")
print(f"Recall       : {rec:.4f}")
print(f"F1-score     : {f1:.4f}")
print(f"Dice         : {dice:.4f}")
print(f"IoU          : {iou:.4f}")
print(f"mAP@50       : {ap50:.4f}")
print("===================================================")

Using device: cuda
Train samples: 4800
Val samples  : 400


Encoder channels: [128, 256, 384]

Training...



Epoch 1/50 | Acc:0.9810 Prec:0.5651 Rec:0.6218 F1:0.5921 Dice:0.5921 IoU:0.4205 mAP50:0.5624


Epoch 2/50 | Acc:0.9875 Prec:0.7344 Rec:0.6840 F1:0.7083 Dice:0.7083 IoU:0.5484 mAP50:0.7481


Epoch 3/50 | Acc:0.9887 Prec:0.7595 Rec:0.7187 F1:0.7385 Dice:0.7385 IoU:0.5855 mAP50:0.7831


...

Epoch 46/50 | Acc:0.9951 Prec:0.8879 Rec:0.8939 F1:0.8909 Dice:0.8909 IoU:0.8032 mAP50:0.9517


Epoch 47/50 | Acc:0.9951 Prec:0.8881 Rec:0.8899 F1:0.8890 Dice:0.8890 IoU:0.8002 mAP50:0.9508


Epoch 48/50 | Acc:0.9951 Prec:0.8869 Rec:0.8936 F1:0.8903 Dice:0.8903 IoU:0.8022 mAP50:0.9529


Epoch 49/50 | Acc:0.9953 Prec:0.8924 Rec:0.8951 F1:0.8937 Dice:0.8937 IoU:0.8079 mAP50:0.9550


Epoch 50/50 | Acc:0.9953 Prec:0.8920 Rec:0.8959 F1:0.8940 Dice:0.8940 IoU:0.8082 mAP50:0.9556

================ FINAL TEST RESULTS ================
Accuracy     : 0.9911
Precision    : 0.8644
Recall       : 0.7843
F1-score     : 0.8224
Dice         : 0.8224
IoU          : 0.6984
mAP@50       : 0.8833
